# 03 · ข้อ 2 ต้นปี 2569 เทียบช่วงเดียวกันของ 2568
ผู้รับผิดชอบ: สมาชิก B · ต้องรัน `01_cleaning.ipynb` (หรือ `python src/run_all.py`) ก่อน

**คำถาม:** ปริมาณผู้ใช้บริการช่วงต้นปี พ.ศ. 2569 เปลี่ยนแปลงจากช่วงเดียวกันของปี พ.ศ. 2568 อย่างไร และวิธีจัดการค่าว่าง/ค่าศูนย์ต่างกันส่งผลต่ออันดับหรือข้อสรุปอย่างไร

โค้ดทั้งหมดอยู่ใน `src/q2_yoy.py` notebook นี้มีหน้าที่**เรียกฟังก์ชันและแสดงผล**พร้อมคำอธิบาย
ฟังก์ชันที่ยังเป็น TODO ให้เขียนใน `src/q2_yoy.py` → ตรวจด้วย `pytest tests/test_q2_yoy.py` → ลบ `#` หน้า cell ที่เกี่ยวข้อง

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import prep
import features
import plots

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
_ = plots.setup_style()
import q2_yoy

df = prep.load_clean()
print(df.shape)

(18705, 18)


## 1. Feature Selection

| Feature ต้นฉบับ | เหตุผล |
|---|---|
| `date`, `year_be` | กำหนดช่วงเทียบที่ตรงกันของสองปี |
| `service`, `purpose`, `mode` | หน่วยที่เปรียบเทียบ |
| `value`, `missing_flag`, `zero_flag`, `zero_long_run` | จำนวน และหลักฐานสำหรับทดสอบวิธีจัดการค่าว่าง/ศูนย์ |

## 2. Data Preparation

In [2]:
end = q2_yoy.comparison_window(df)
rows = q2_yoy.select_rows(df)
win = rows.drop_duplicates(["year_be", "date"]).groupby("year_be")["date"]
display(pd.DataFrame({"first": win.min(), "last": win.max(), "days": win.nunique(),
                      "weekend_days": win.apply(lambda d: (d.dt.dayofweek >= 5).sum())}))
print("ช่วงเทียบ: 1 ม.ค. ถึง (เดือน, วัน) =", end, "·", rows["service"].nunique(), "services")
pd.Series(q2_yoy.SCENARIOS, name="วิธีจัดการค่าว่าง/ศูนย์")

,first,last,days,weekend_days
year_be,,,,
2568,2025-01-01,2025-03-11,70,20
2569,2026-01-01,2026-03-11,70,20


ช่วงเทียบ: 1 ม.ค. ถึง (เดือน, วัน) = (3, 11) · 39 services


S1                 ตัด NaN ออก, เก็บศูนย์ไว้เป็นค่าจริง
S2    ศูนย์ที่อยู่ในช่วงศูนย์ยาว (zero_long_run) ถือ...
S3    Paired days: ใช้เฉพาะวันที่ทั้งสองปีมีค่าที่ไม...
S4    เติม NaN ด้วย median ของวันเดียวกันในสัปดาห์ เ...
S5                  แทน NaN ด้วย 0 (ตัวอย่างวิธีที่ผิด)
Name: วิธีจัดการค่าว่าง/ศูนย์, dtype: str

## 3. Feature Engineering

| Feature ใหม่ | สูตร / เงื่อนไข | ฟังก์ชัน |
|---|---|---|
| `avg_daily_68`, `avg_daily_69` | ค่าเฉลี่ยต่อวันในช่วงเทียบ | `q2_yoy.yoy_table` (TODO) |
| `yoy_change_pct` | `features.safe_pct_change(avg_daily_69, avg_daily_68)` | `q2_yoy.yoy_table` (TODO) |
| `abs_change` | `avg_daily_69 − avg_daily_68` | `q2_yoy.yoy_table` (TODO) |
| `coverage_68/69` | วันที่มีค่าใช้ได้ / วันในช่วง | `q2_yoy.yoy_table` (TODO) |
| `rank`, `spearman_vs_reference` | อันดับภายใต้ S1–S5 และความสอดคล้องกับ S1 | `q2_yoy.run_scenarios`, `rank_stability` (TODO) |

In [3]:
# table = q2_yoy.yoy_table(q2_yoy.apply_scenario(rows, "S2"))
# sensitivity = q2_yoy.run_scenarios(rows)
# display(table.sort_values("yoy_change_pct"), q2_yoy.rank_stability(sensitivity))

## 4. Visualization
กราฟที่วางไว้: ① `plot_yoy_bars` ② `plot_rank_by_scenario`

In [4]:
# fig = q2_yoy.plot_yoy_bars(table); plots.save_fig(fig, "q2_yoy_bars"); plt.show()
# fig = q2_yoy.plot_rank_by_scenario(sensitivity); plots.save_fig(fig, "q2_rank_by_scenario"); plt.show()

**การตีความ:** _(เขียนจากตัวเลขจริงในกราฟ)_

## 5. Analysis & Findings
- คำตอบของคำถาม:
- ข้อค้นพบสำคัญ:
- ข้อจำกัดของข้อมูล:

## 6. บันทึก Prepared Dataset

In [5]:
# table.to_csv(prep.PREPARED_DIR / q2_yoy.OUTPUT_FILE, index=False, encoding="utf-8-sig")
# sensitivity.to_csv(prep.PREPARED_DIR / q2_yoy.SENSITIVITY_FILE, index=False, encoding="utf-8-sig")